In [22]:
import pandas as pd

### 1.1 Chargement des données¶

In [24]:
df = pd.read_csv('/Users/bakarybengaly/Documents/fluxapp_projet_clean/customers.csv')
df.head()

,customer_id,plan,acquisition_channel,country,age,tenure_months,monthly_spend_eur,sessions_last30d,support_tickets,churned
0,1,Pro,Ads,FR,19.0,30,26.01,16,0,1
1,2,Business,Referral,CA,47.0,16,81.05,21,2,0
2,3,Pro,Organic,CH,38.0,27,26.19,10,1,1
3,4,Basic,Organic,CA,44.0,31,10.79,10,0,0
4,5,Basic,Organic,Autre,64.0,12,11.09,8,1,1


### 1.2 Exploration initiale¶

In [26]:
df.shape

(1500, 10)

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          1500 non-null   int64  
 1   plan                 1500 non-null   object 
 2   acquisition_channel  1500 non-null   object 
 3   country              1500 non-null   object 
 4   age                  1480 non-null   float64
 5   tenure_months        1500 non-null   int64  
 6   monthly_spend_eur    1480 non-null   float64
 7   sessions_last30d     1500 non-null   int64  
 8   support_tickets      1500 non-null   int64  
 9   churned              1500 non-null   int64  
dtypes: float64(2), int64(5), object(3)
memory usage: 117.3+ KB


### 1.3 Traitement des valeurs manquantes¶

In [29]:
age_mediane = df['age'].median()
print(f"Médiane de l'âge : {age_mediane}")


Médiane de l'âge : 41.0


In [30]:
df['age'] = df['age'].fillna(age_mediane)
print("valeur manquante de l'âge est" ,df['age'].isnull().sum())

valeur manquante de l'âge est 0


In [31]:
mediane_par_plan = df.groupby('plan')['monthly_spend_eur'].median()
print(mediane_par_plan)

plan
Basic        9.100
Business    79.160
Pro         28.755
Name: monthly_spend_eur, dtype: float64


In [32]:
df['monthly_spend_eur'] = df.groupby('plan')['monthly_spend_eur'].transform(
    lambda x: x.fillna(x.median())
)
print("Valeurs manquantes restantes dans monthly_spend_eur", df['monthly_spend_eur'].isnull().sum())

Valeurs manquantes restantes dans monthly_spend_eur 0


In [33]:
df.isnull().sum()

customer_id            0
plan                   0
acquisition_channel    0
country                0
age                    0
tenure_months          0
monthly_spend_eur      0
sessions_last30d       0
support_tickets        0
churned                0
dtype: int64

### 1.3 Statistiques descriptives

In [43]:
df.describe()

,customer_id,age,tenure_months,monthly_spend_eur,sessions_last30d,support_tickets,churned
count,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.00000,1500.000000
mean,750.500000,41.214667,18.116000,24.605913,12.973333,1.25000,0.387333
std,433.157015,13.600879,10.192275,23.192640,4.065584,1.10347,0.487303
min,1.000000,18.000000,1.000000,5.000000,3.000000,0.00000,0.000000
25%,375.750000,30.000000,9.000000,8.830000,10.000000,0.00000,0.000000
50%,750.500000,41.000000,18.000000,12.800000,13.000000,1.00000,0.000000
75%,1125.250000,53.000000,27.000000,29.882500,15.000000,2.00000,1.000000
max,1500.000000,64.000000,35.000000,85.760000,27.000000,6.00000,1.000000


In [53]:
print("Taux de churn global est :", round(df['churned'].mean() * 100, 1), "%")
print("Nombre de clients churned est :", df['churned'].sum())
print("Nombre de clients actifs est :", (df['churned'] == 0).sum())

Taux de churn global est : 38.7 %
Nombre de clients churned est : 581
Nombre de clients actifs est : 919


### 1.4 Analyses croisées

In [57]:
print("Taux de churn par plan :")
print((df.groupby('plan')['churned'].mean() * 100).round(1))

Taux de churn par plan :
plan
Basic       46.3
Business    20.6
Pro         33.1
Name: churned, dtype: float64


In [59]:
print("Taux de churn par canal :")
print((df.groupby('acquisition_channel')['churned'].mean() * 100).round(1))

Taux de churn par canal :
acquisition_channel
Ads         43.2
Organic     38.3
Referral    37.9
Social      32.8
Name: churned, dtype: float64


In [61]:
df['tenure_bin'] = pd.cut(df['tenure_months'], 
                           bins=[0, 6, 12, 24, 60],
                           labels=['0-6 mois', '7-12 mois', '13-24 mois', '25+ mois'])

print("Taux de churn par ancienneté :")
print((df.groupby('tenure_bin', observed=True)['churned'].mean() * 100).round(1))

Taux de churn par ancienneté :
tenure_bin
0-6 mois      48.6
7-12 mois     47.0
13-24 mois    37.3
25+ mois      30.3
Name: churned, dtype: float64
